In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
import os
import pandas as pd
import gdown
import numpy as np
import torch
import gc
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import ViTImageProcessor, ViTModel


In [5]:
input_folders = os.listdir('/kaggle/input')
print("Daftar folder yang tersedia di /kaggle/input:")
print(input_folders)

Daftar folder yang tersedia di /kaggle/input:
['datasets']


In [6]:
base_input = '/kaggle/input/datasets'
isi_datasets = os.listdir(base_input)
print(f"Isi folder datasets: {isi_datasets}")

Isi folder datasets: ['fati22']


In [7]:
path_username = '/kaggle/input/datasets/fati22'
isi_fati22 = os.listdir(path_username)
print(f"Isi di dalam folder fati22: {isi_fati22}")

Isi di dalam folder fati22: ['tokopedia-images-product']


In [8]:
dataset_path = '/kaggle/input/datasets/fati22/tokopedia-images-product'
try:
    total_gambar = len(os.listdir(dataset_path))
    print(f"Total gambar yang terdeteksi: {total_gambar:,}")
except FileNotFoundError:
    print("Path tidak ditemukan, pastikan penulisan path sudah benar.")
except Exception as e:
    print(f"Terjadi kesalahan: {e}")

Total gambar yang terdeteksi: 3,475,088


In [11]:
existing_images = {f.replace('.jpg', '') for f in os.listdir(dataset_path) if f.endswith('.jpg')}
print(f"Total ID Product yang ditemukan dalam folder: {len(existing_images):,}")

Total ID Product yang ditemukan dalam folder: 3,475,088


In [9]:
file_id = '1CK5x5i5p1xrz4cKT_bJJY_N-9_C5zWuh'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'data_download.csv'
gdown.download(url, output, quiet=False)
df = pd.read_csv(output, sep=',', on_bad_lines='skip')

Downloading...
From (original): https://drive.google.com/uc?id=1CK5x5i5p1xrz4cKT_bJJY_N-9_C5zWuh
From (redirected): https://drive.google.com/uc?id=1CK5x5i5p1xrz4cKT_bJJY_N-9_C5zWuh&confirm=t&uuid=c35c7b45-cef3-44a0-be7a-e6157ddd919e
To: /kaggle/working/data_download.csv
100%|██████████| 926M/926M [00:05<00:00, 175MB/s]  


In [12]:
df.head()

,ID_Product,Shop_Name,Product_Name,Price,Rating,URL_Product
0,1,Awkward Brand,Kaos Polos Anak Katun Combed 30s Hijau Olive,Rp74.000,NaN,https://www.tokopedia.com/awkwardofficial/kaos...
1,2,Awkward Brand,Kaos Polos Anak Katun Combed 30s Biru Navy,Rp74.000,5.0,https://www.tokopedia.com/awkwardofficial/kaos...
2,3,Awkward Brand,Kaos Polos Anak Katun Combed 30s Kuning Mustard,Rp74.000,NaN,https://www.tokopedia.com/awkwardofficial/kaos...
3,4,Awkward Brand,Kaos Polos Anak Katun Combed 30s Merah Maroon,Rp74.000,5.0,https://www.tokopedia.com/awkwardofficial/kaos...
4,5,Awkward Brand,Kaos Polos Anak Katun Combed 30s Abu-abu Misty,Rp74.000,5.0,https://www.tokopedia.com/awkwardofficial/kaos...


In [13]:
len(existing_images)

3475088

In [14]:
df_final = df[df["ID_Product"].astype(str).isin(existing_images)].copy()
df_final.shape

(3475149, 6)

In [16]:
duplikat = df[df.duplicated(subset=['ID_Product'], keep=False)]
print(f"Jumlah baris duplikat: {len(duplikat)}")

Jumlah baris duplikat: 122


In [15]:
df_final = df_final.drop_duplicates(subset=['ID_Product'], keep='first')
df_final.shape

(3475088, 6)

In [20]:
class TokopediaDataset(Dataset):
    def __init__(self, dataframe, img_dir, processor):
        self.df = dataframe
        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        id_product = row['ID_Product']
        judul = row['Product_Name']
        
        # Sesuai skema jenius Anda: ID 1 -> 1.jpg
        img_name = f"{int(id_product)}.jpg"
        img_path = os.path.join(self.img_dir, img_name)
        
        try:
            image = Image.open(img_path).convert("RGB")
            inputs = self.processor(images=image, return_tensors="pt")
            pixel_values = inputs['pixel_values'].squeeze(0)
        except Exception:
            # Jika gambar tidak ada/rusak, beri tensor nol agar urutan tetap sinkron
            pixel_values = torch.zeros(3, 224, 224)
            
        return pixel_values, str(id_product), judul

In [18]:
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = ViTModel.from_pretrained("google/vit-base-patch16-224").cuda()
model.eval()

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): ViTOutput(
          (d

In [21]:
chunk_split = [
    (0, 500000, 1),
    (500000, 1000000, 2),
    (1000000, 1500000, 3),
    (1500000, 2000000, 4),
    (2000000, 2500000, 5),
    (2500000, 3000000, 6),
    (3000000, None, 7) 
]

for start, end, chunk_num in chunk_split:
    if end:
        df_chunk = df_final.iloc[start:end].copy()
    else:
        df_chunk = df_final.iloc[start:].copy()
        
    dataset = TokopediaDataset(df_chunk, dataset_path, processor)
    dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=0)
    
    all_vectors, all_ids, all_juduls = [], [], []
    
    with torch.no_grad():
        for i, (imgs, ids, juduls) in enumerate(dataloader):
            imgs = imgs.cuda()
            outputs = model(pixel_values=imgs)
            embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_vectors.append(embeddings)
            all_ids.extend(ids)
            all_juduls.extend(juduls)
            if i % 200 == 0:
                print(f"Chunk {chunk_num} -> Batch {i} selesai, flush=True")
                
    df_output = pd.DataFrame({
        'ID_Product': all_ids,
        'Judul': all_juduls,
        'Embedding': np.vstack(all_vectors).tolist()
    })
    
    nama_file = f"chunk_{chunk_num}.parquet"
    df_output.to_parquet(nama_file, engine='pyarrow')
    print(f"Chunk {chunk_num} selesai")
    
    del df_chunk, dataset, dataloader, df_output, all_vectors, all_ids, all_juduls
    gc.collect()
    torch.cuda.empty_cache()

Chunk 1 -> Batch 0 selesai, flush=True
Chunk 1 -> Batch 200 selesai, flush=True


KeyboardInterrupt: 